In [11]:
import os

if not os.path.exists("SQD"):
    !git clone https://github.com/KhushiPandey1805/SQD.git SQD
    print("Cloned fresh.")
else:
    os.chdir("SQD")
    !git pull
    os.chdir("/content")
    print("Already present, pulled latest changes.")

Cloning into 'SQD'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 37 (delta 11), reused 33 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 174.40 KiB | 5.45 MiB/s, done.
Resolving deltas: 100% (11/11), done.
Cloned fresh.


In [12]:
!find /content/SQD -iname "*HERMITIAN*" 2>/dev/null

/content/SQD/hamiltonian/H2O_CAS4e4o_JW_Pauli_Hamiltonian_HERMITIAN.txt


In [1]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp

In [2]:
def parse_hamiltonian_file(filepath: str, num_qubits: int) -> SparsePauliOp:
    """Read a Pauli-list Hamiltonian text file and return a SparsePauliOp."""
    pauli_list = []

    with open(filepath, "r") as f:
        for raw_line in f:
            line = raw_line.strip()

            # Skip blank lines and comment lines (metadata headers).
            if not line or line.startswith("#"):
                continue

            # Split off the coefficient (the last ":"-separated field).
            term_str, coeff_str = line.rsplit(":", 1)
            coeff = float(coeff_str.strip())
            term_str = term_str.strip()

            # Start every qubit as Identity, then overwrite the ones
            # explicitly mentioned in this term.
            paulis = ["I"] * num_qubits

            if term_str != "I":
                for token in term_str.split():
                    letter = token[0]          # 'X', 'Y', or 'Z'
                    qubit_index = int(token[1:])  # the number after it
                    paulis[qubit_index] = letter

            # Qiskit's Pauli-label convention reads left-to-right as
            # qubit (num_qubits-1) ... qubit 0 (i.e. qubit 0 is the
            # *rightmost* character). Our `paulis` list is indexed the
            # opposite way (paulis[0] = qubit 0), so reverse it.
            label = "".join(reversed(paulis))

            pauli_list.append((label, coeff))

    hamiltonian = SparsePauliOp.from_list(pauli_list)

    # Combine any duplicate Pauli strings and drop exact-zero terms.
    hamiltonian = hamiltonian.simplify()

    return hamiltonian

In [3]:
def check_hermitian(hamiltonian: SparsePauliOp) -> bool:
    """Verify H = H^dagger (a physical requirement for any Hamiltonian)."""
    matrix = hamiltonian.to_matrix()
    return np.allclose(matrix, matrix.conj().T)

In [4]:
def exact_ground_state_energy(hamiltonian: SparsePauliOp) -> float:
    """
    Brute-force exact diagonalization. Only reasonable for small qubit
    counts (here: 8 qubits -> 256x256 matrix, trivially fast) — this is
    NOT what you'd do for a real SQD-scale problem, it's purely a sanity
    check that our parsing is correct.
    """
    matrix = hamiltonian.to_matrix()
    eigenvalues = np.linalg.eigvalsh(matrix)
    return eigenvalues[0]

In [13]:
if __name__ == "__main__":
    NUM_QUBITS = 8
    FILEPATH = "/content/SQD/hamiltonian/H2O_CAS4e4o_JW_Pauli_Hamiltonian_HERMITIAN.txt"
    REFERENCE_ENERGY = -76.11669094458742  # from the file's header comment

    hamiltonian = parse_hamiltonian_file(FILEPATH, NUM_QUBITS)

    print(f"Number of qubits: {hamiltonian.num_qubits}")
    print(f"Number of Pauli terms after simplify(): {len(hamiltonian.paulis)}")
    print(f"Is Hermitian: {check_hermitian(hamiltonian)}")

    ground_energy = exact_ground_state_energy(hamiltonian)
    print(f"Exact diagonalization ground-state energy: {ground_energy}")
    print(f"Reference energy (from file header):        {REFERENCE_ENERGY}")
    print(f"Difference: {abs(ground_energy - REFERENCE_ENERGY):.10f} Ha")

Number of qubits: 8
Number of Pauli terms after simplify(): 337
Is Hermitian: True
Exact diagonalization ground-state energy: -76.11669094459984
Reference energy (from file header):        -76.11669094458742
Difference: 0.0000000000 Ha
